# Durable Alignment in LLMs via Orthogonal Memory — v2

**Companion notebook to the paper _Durable Alignment via Orthogonal Memory in Frozen Language Models_.**

This is the **claim-driven rebuild** (v2) of the original organic-research notebook. The eight
claims are organised into a four-layer dependency stack (Existence → Properties → Operations →
Composition); each section produces one piece of empirical evidence and writes its artifacts
to `mmlu_ibf_out/` under the `cN_*` naming scheme.

## Central thesis

> _IBF is a local durable alignment substrate for frozen LLMs whose properties (decoupling,
> generality, distinction) support a clean operational lifecycle (install / revise / remove
> with preserved locality) which in turn supports complementary deductive and inductive
> composition paths — without modifying base-model weights._

## Four-layer stack

```
Layer 4 — COMPOSITION
   C7  Compiled closure (deductive)
   C8  Discovery-driven extension (inductive)
        — includes Crucible-adjudicated conflict resolution (D8 evidence)

Layer 3 — OPERATIONS
   C5  Truth-maintenance lifecycle
   C6  Locality preservation

Layer 2 — PROPERTIES
   C2  Substrate decoupling under base evolution
   C3  Cross-model mechanism generality
   C4  Distinction from kNN/RAG

Layer 1 — EXISTENCE
   C1  Local durable alignment without weight editing
```

Each upper layer is _earned_ by the layers below it. Falsifying a lower claim cascades upward.

## Foundational anchoring

Each claim in this notebook is the LLM-substrate realisation of a concept from the
foundational paper *Information as Alignment v1* (Sections 2–4, 5.4, 8.1). The mapping:

| Companion claim | Foundational anchor |
|---|---|
| C1 — Existence | Foundational Claim 1 (Memory) |
| C2 — Decoupling | Postulate 1 constraint (iii) |
| C3 — Generality | Foundational Claim 5 (Discrete Convergence) |
| C4 — Distinction | Foundational § 8.2 |
| C5 — Lifecycle | Foundational Claims 1 + 4 (Memory + Self-Correction) |
| C6 — Locality | Postulate 1 localisation kernel `K(y, x_S)` |
| C7 — Deductive composition | Foundational § 7.2 + § 5.4 LLM-extension flag |
| C8 — Inductive composition | Foundational Claims 2 + 3 (Agency + Intelligence) + § 8.1 regime-dependence |

## Reading order

Cells are intended to be run top to bottom. Setup (S1–S3) builds the engine, geometry, and
dataset; the eight claim sections each consume those globals and write their artifacts; the
two synthesis sections (S4–S5) aggregate the JSON outputs into a single paper-deliverable.

The previous organic notebook `(IBF)Companion-LLM-Durable-Alignment.ipynb` is preserved as
the paper-run historical record. This v2 is the canonical artifact going forward.


## Run configuration

`RUN_MODE` controls every long-running cell. Three values are supported:

- `"smoke"` — minimum epochs, small datasets, smoke caps. ~30 min end-to-end on an A100.
- `"paper"` — full paper-grade run with standard hard caps. ~35h+ end-to-end.
- `"verify-convergence"` — paper-grade caps **with** the strong-convergence early-stop
  protocol enabled (see Part 6 of the handover). This is the C1 validation-gate mode.
  Estimated ~2.5× compute reduction if the gate passes.

Headline numbers below are the C1 validation-gate references — used in every
long-running cell's `EXPECTED / GOT / WITHIN_TOLERANCE` log line.


In [ ]:
# ════════════════════════════════════════════════════════════════
# Top-level run configuration
# ════════════════════════════════════════════════════════════════
# RUN_MODE: one of {"smoke", "paper", "verify-convergence"}.
# SEED: global seed; each long-running cell uses SEED + offset.
# ════════════════════════════════════════════════════════════════

import os, datetime

RUN_MODE = os.environ.get("IBF_RUN_MODE", "paper")
assert RUN_MODE in ("smoke", "paper", "verify-convergence"), f"Bad RUN_MODE: {RUN_MODE}"

SEED = 42

# Per-claim seed offsets — keeps each long-running cell deterministic and independent.
SEED_OFFSETS = {
    "C1": 100, "C2": 200, "C3": 300, "C4": 400,
    "C5": 500, "C6": 600, "C7": 700, "C8": 800,
}

# C1 validation-gate reference (handover Part 6).
# avg lin from the paper-run canonical_training_results.json.
HEADLINE_AVG_LIN = 0.954
HEADLINE_AVG_LIN_TOL = 0.01  # gate threshold: ±0.01 absolute

# Convergence-stop protocol — gated by RUN_MODE.
EARLY_STOP_STRONG_CONVERGE = (RUN_MODE == "verify-convergence")

# Pin run identifier for log/artifact naming.
RUN_ID = os.environ.get(
    "IBF_RUN_ID",
    datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
)

print(f"RUN_MODE                       = {RUN_MODE!r}")
print(f"SEED                           = {SEED}")
print(f"EARLY_STOP_STRONG_CONVERGE     = {EARLY_STOP_STRONG_CONVERGE}")
print(f"HEADLINE_AVG_LIN               = {HEADLINE_AVG_LIN} ± {HEADLINE_AVG_LIN_TOL}")
print(f"RUN_ID                         = {RUN_ID}")


# Part I — Setup (S1 / S2 / S3)

Construct the deterministic environment: the universal engine (with the Reading C agency
patch), the representation geometry, and the Future Industries dataset. After Part I, every
subsequent claim section consumes the same `ibf` object, `precomputed` dict, and
`phase_data` structure.

---

## § S1 — Universal IBF engine (with Reading C patch)

**Objective.** Define the universal IBF engine as a particle approximation of the
foundational paper's Modification Postulate (Section 4 of the foundation paper). The engine
is **domain-agnostic**: it operates on a d-dimensional latent space supplied by a frozen
encoder and a baseline evaluator. Every modification dynamic below corresponds to a specific
equation or constraint in Postulate 1.

**Mapping to Postulate 1.**

| Engine method | Postulate 1 component |
|---|---|
| `MemoryCenter` | particle `c_i = (z_i, a_i, σ_i, μ_eff,i, ctx_i, …)` (foundation § 4.1) |
| `kernel_batch` | localisation kernel `K(y, x_S) = exp(-‖y - x_S‖² / 2σ²)` (Postulate 1, constraint i) |
| `_read_gate` | reflexive read-gating `γ_i` (foundation § 4.2, Eq. 9) |
| `delta_R` | kernel readout `δR̂(z) = Σ γ_i v_i K(z, z_i)` (Eq. 7) |
| `delta_k` | **intensive** responsiveness readout `δk(z) = Σ I·γ·w·K / Σ I·γ·K` (Eq. 8) |
| `_update_value` | write path Pass 2, same-context spatial learning (Eq. 11) |
| `_update_agency` | parallel equation for responsiveness modification (Postulate 1) |
| `_crystallize` | stability transition `μ_eff → μ_cryst` under exposure + convergence |
| `_crucible` | contradiction-triggered dissolution `v_i · D̄_raw < θ_rev` (Eq. 13) |
| `_merge_pop` | merge + capacity control under dynamic spatial threshold |

### Reading C patch (mandatory)

The original engine gates the agency channel's `w` update and `δk` readout on
crystallisation. This is a conservative discretisation that under-implements Postulate 1's
**parallel equation** framing (foundation § 3.3, where the paper explicitly notes _"A
parallel equation governs the responsiveness modification δk_S (the agency channel)"_).

The Reading C refinement gates the agency channel on **history sufficiency**
(`len(c.D_history) ≥ n_agency_min`) rather than on crystallisation. This brings the agency
channel into structural parity with the value channel: transient `w` updates with a slower
learning rate (`eta_k_cryst`), and transient `w` contributes to `δk` readout. The change is
behaviourally inert on the large-data regimes from the foundational paper (chess / RRW /
CIFAR — see foundation § 8.1) where crystallisation fires within thousands of updates, but
it is the only way to make the agency channel testable on the small-data LLM substrate.

**Empirical justification (D6 result).**

Under the patched engine, the β k_eff at A→C queries traces a U-shape (4.66 → 2.58 → 3.23)
tracking the local D-variance dynamics, while the α (status-quo) k_eff stays flat at the
baseline `k_0 = 5.000`. This U-shape is the empirical signature of Postulate 1's parallel
equation: responsiveness rises in low-variance regions (where corrections converge) and
drops in high-variance regions (where alignment is uncertain). See
`AGENCY_DISCRETIZATION_NOTE.md` and `mmlu_ibf_out/fi_agency_channel_d6_alpha_vs_beta.json`
for the supervisor's pre-merge validation transcript.

**Pre-merge validation (handover Part 7).** C1 with the patched engine must match unpatched
within ±0.01 on avg lin; C3 (Qwen) must match within ±0.02 on all 6 cross-model metrics.
Both gates are checked downstream in this notebook.


In [ ]:
# ════════════════════════════════════════════════════════════════
# § S1 — Universal IBF engine (with Reading C patch)
# Layer: setup
# Presupposes: nothing
# Artifacts: none (defines IBFParams, MemoryCenter, IBFEngine)
# Convergence-stop: n/a
# ════════════════════════════════════════════════════════════════
# This is the domain-agnostic engine that instantiates Postulate 1 of the
# foundation paper. Every modification dynamic below maps to a specific
# equation in foundation § 4.
#
# Reading C patch (mandatory): the agency channel is gated by history
# sufficiency rather than crystallisation. See the markdown above for the
# full justification.
# ════════════════════════════════════════════════════════════════

import torch, torch.nn.functional as F, numpy as np
import time, os, json, random
from dataclasses import dataclass, field
from typing import List, Dict
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
OUT_DIR = "mmlu_ibf_out"; os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Versioned engine identifier — written into every C-cell artifact's metadata.
ENGINE_VERSION = "2.0-history_gate"


@dataclass
class IBFParams:
    sigma: float = 0.206
    merge_threshold: float = 0.366
    sigma_agency: float = None
    eta: float = 0.1
    eta_cryst: float = 0.005
    eta_k: float = 0.05
    # Reading C addition: parallel learning rate for crystallised agency.
    eta_k_cryst: float = 0.005
    mu_base: float = 0.06
    mu_crystallized: float = 0.001
    v_max: float = 0.50
    w_max: float = 5.0
    k_0: float = 5.0
    k_min: float = 1.0
    crystallization_threshold: int = 15
    convergence_threshold: float = 0.025
    # Reading C addition: history sufficiency gate for agency channel.
    n_agency_min: int = 20
    n_cross_min: int = 8
    reversal_threshold: float = -0.0125
    w_dvar_threshold: float = 0.005
    activation_threshold: float = 0.01
    creation_threshold: float = 0.01
    capacity: int = 5000
    alpha_shrink: float = 100.0
    sigma_floor: float = 0.15
    min_samples_shrink: int = 50

    @classmethod
    def from_calibration(cls, d):
        return cls(sigma=d["SIGMA"], merge_threshold=d["MERGE_THRESHOLD"],
                   v_max=d.get("V_MAX", 0.5), capacity=d.get("DR_CAPACITY", 5000))


@dataclass
class MemoryCenter:
    z: np.ndarray
    v: float = 0.0
    w: float = 0.0
    n_updates: int = 0
    D_sum: float = 0.0
    D_sq_sum: float = 0.0
    mu_eff: float = 0.06
    context_id: int = -1
    birth_epoch: int = 0
    context_update_counts: Dict[int, int] = field(default_factory=dict)
    sigma: float = 0.58
    D_history: List[float] = field(default_factory=list)
    D_history_cross: List[float] = field(default_factory=list)
    was_ever_crystallized: bool = False
    crucible_verified: bool = False
    dissolution_log: List[Dict] = field(default_factory=list)

    def is_crystallized(self):
        return self.mu_eff < 0.06 - 1e-6

    def D_var_rolling(self):
        if len(self.D_history) < 25:
            return 0.0
        r = self.D_history[20:][-50:]
        return float(np.var(r)) if len(r) >= 5 else 0.0


class IBFEngine:
    """Universal IBF engine — discrete particle approximation of Postulate 1.

    The engine is domain-agnostic: it operates on a d-dimensional latent space
    supplied by a frozen encoder and a baseline evaluator. Reading C patch is
    applied: the agency channel gates on history sufficiency rather than on
    crystallisation, bringing it into parity with the value channel and matching
    Postulate 1's parallel-equation framing.
    """

    def __init__(self, params=None):
        self.p = params or IBFParams()
        self.value_centers, self.agency_centers = [], []
        self.current_epoch = 0
        self.current_context_id = -1
        self._epoch_D_vals = []
        self._merge_ratio = self.p.merge_threshold / self.p.sigma
        self._dynamic_sigma_floor = min(self.p.sigma_floor, self.p.sigma * 0.25)
        self._needle_threshold = self.p.sigma * 0.50
        self._sigma_agency = self.p.sigma_agency if self.p.sigma_agency else self.p.sigma
        self._D_running_sum = 0.0
        self._D_running_count = 0
        print(f"IBF: σ={self.p.sigma:.4f}, cap={self.p.capacity}, engine_version={ENGINE_VERSION}")

    def kernel_batch(self, z, centers):
        if not centers:
            return np.array([])
        za = np.array([c.z for c in centers])
        sq = np.sum((za - z[None, :]) ** 2, 1)
        return np.exp(-sq / (2 * np.array([c.sigma for c in centers]) ** 2))

    def _read_gate(self, c):
        if c.context_id == self.current_context_id:
            return 1.0
        return 1.0 if c.is_crystallized() and c.crucible_verified else 0.0

    def delta_R(self, z):
        if not self.value_centers:
            return 0.0
        K = self.kernel_batch(z, self.value_centers); t = 0.0
        for i, c in enumerate(self.value_centers):
            g = self._read_gate(c)
            if g > 0 and K[i] > self.p.activation_threshold:
                t += g * c.v * K[i]
        return t

    def delta_k(self, z):
        """Intensive responsiveness readout (foundation § 4.2, Eq. 8).

        Reading C patch: gate on history sufficiency, not crystallisation.
        """
        if not self.agency_centers:
            return 0.0
        K = self.kernel_batch(z, self.agency_centers); tw = sk = 0.0
        for i, c in enumerate(self.agency_centers):
            # PATCHED: history-sufficiency gate replaces crystallisation gate.
            if len(c.D_history) < self.p.n_agency_min:
                continue
            g = self._read_gate(c)
            if g > 0 and K[i] > self.p.activation_threshold:
                tw += g * c.w * K[i]; sk += g * K[i]
        return tw / sk if sk > 1e-6 else 0.0

    def k_eff(self, z):
        return max(self.p.k_min, self.p.k_0 + self.delta_k(z))

    def compute_D_and_update(self, board_before, board_after_own_move,
                             board_after_opponent, move_chosen=None,
                             move_opponent=None, R_imposed_override=None):
        z_before = RC_encode(board_before)
        z_chosen = (RC_encode_move(board_before, board_after_own_move, move_chosen)
                    if move_chosen is not None else RC_encode(board_after_own_move))
        R_f = RC_R_field(board_after_own_move)
        dR = self.delta_R(z_chosen)
        R_eff_ch = np.clip(1.0 - (R_f + dR), 0, 1)
        R_eff_imp = float(R_imposed_override) if R_imposed_override is not None else 0.5
        D = R_eff_imp - R_eff_ch
        self._D_running_count += 1; self._D_running_sum += D
        D -= self._D_running_sum / self._D_running_count
        self._epoch_D_vals.append(D)
        self._update_value(z_chosen, D)
        self._update_agency(z_before, D)
        return {"D": D, "R_eff_chosen": float(R_eff_ch), "dR_chosen": float(dR)}

    def _shrink(self, c):
        if len(c.D_history) >= self.p.min_samples_shrink:
            dv = c.D_var_rolling()
            c.sigma = min(c.sigma, max(self._dynamic_sigma_floor,
                          self.p.sigma / (1 + self.p.alpha_shrink * dv)))

    def _update_value(self, z, D):
        neg_D = -D; ctx = self.current_context_id
        for c in [c for c in self.value_centers if c.is_crystallized() and c.context_id != ctx]:
            kw = float(self.kernel_batch(z, [c])[0])
            if kw < self.p.activation_threshold:
                continue
            c.v = np.clip(c.v + self.p.eta_cryst * neg_D * kw, -self.p.v_max, self.p.v_max)
            c.n_updates += 1
            c.context_update_counts[ctx] = c.context_update_counts.get(ctx, 0) + 1
            c.D_history_cross.append(neg_D)
        sc = [c for c in self.value_centers if c.context_id == ctx]
        if sc:
            Ks = self.kernel_batch(z, sc); mx = float(np.max(Ks))
        else:
            Ks = np.array([]); mx = 0.0
        if mx < self.p.creation_threshold:
            if len(self.value_centers) < self.p.capacity:
                nc = MemoryCenter(
                    z=z.copy(),
                    v=np.clip(self.p.eta * neg_D, -self.p.v_max, self.p.v_max),
                    mu_eff=self.p.mu_base, context_id=ctx,
                    birth_epoch=self.current_epoch, sigma=self.p.sigma,
                )
                nc.n_updates = 1; nc.D_sum = neg_D; nc.D_sq_sum = neg_D ** 2
                nc.D_history.append(neg_D); nc.context_update_counts[ctx] = 1
                self.value_centers.append(nc)
            return
        for i, c in enumerate(sc):
            kw = float(Ks[i])
            if kw < self.p.activation_threshold:
                continue
            lr = self.p.eta_cryst if c.is_crystallized() else self.p.eta
            c.v = np.clip(c.v + lr * neg_D * kw, -self.p.v_max, self.p.v_max)
            c.n_updates += 1; c.D_sum += neg_D * kw; c.D_sq_sum += (neg_D * kw) ** 2
            c.context_update_counts[ctx] = c.context_update_counts.get(ctx, 0) + 1
            c.D_history.append(neg_D * kw); self._shrink(c)

    def _update_agency(self, z, D):
        """Responsiveness modification — parallel equation to δR (Postulate 1).

        Reading C patch: history-sufficiency gates (`len(c.D_history) >=
        n_agency_min`) replace the crystallisation gate, and a parallel
        learning rate `eta_k_cryst` matches `eta_cryst` for the value channel.
        """
        ctx = self.current_context_id
        for c in [c for c in self.agency_centers if c.is_crystallized() and c.context_id != ctx]:
            kw = float(self.kernel_batch(z, [c])[0])
            if kw < self.p.activation_threshold:
                continue
            c.n_updates += 1
            c.context_update_counts[ctx] = c.context_update_counts.get(ctx, 0) + 1
            c.D_history_cross.append(D)
        sc = [c for c in self.agency_centers if c.context_id == ctx]
        if sc:
            Ks = self.kernel_batch(z, sc); mx = float(np.max(Ks))
        else:
            Ks = np.array([]); mx = 0.0
        if mx < self.p.creation_threshold:
            if len(self.agency_centers) < self.p.capacity:
                nc = MemoryCenter(
                    z=z.copy(), mu_eff=self.p.mu_base, context_id=ctx,
                    birth_epoch=self.current_epoch, sigma=self._sigma_agency,
                )
                nc.n_updates = 1; nc.D_sum = D; nc.D_sq_sum = D ** 2
                nc.D_history.append(D); nc.context_update_counts[ctx] = 1
                self.agency_centers.append(nc)
            return
        for i, c in enumerate(sc):
            kw = float(Ks[i])
            if kw < self.p.activation_threshold:
                continue
            c.n_updates += 1; c.D_sum += D * kw; c.D_sq_sum += (D * kw) ** 2
            c.context_update_counts[ctx] = c.context_update_counts.get(ctx, 0) + 1
            c.D_history.append(D * kw)
            # PATCHED: history-sufficiency gate; parallel learning rates.
            if len(c.D_history) >= self.p.n_agency_min:
                dv = c.D_var_rolling()
                tw = np.clip(self.p.w_max * (1 - dv / self.p.w_dvar_threshold),
                             -self.p.w_max, self.p.w_max)
                lr = self.p.eta_k_cryst if c.is_crystallized() else self.p.eta_k
                c.w += lr * kw * (tw - c.w)
                c.w = np.clip(c.w, -self.p.w_max, self.p.w_max)
            self._shrink(c)

    def _crystallize(self):
        for pop in [self.value_centers, self.agency_centers]:
            for c in pop:
                if c.is_crystallized() or c.n_updates < self.p.crystallization_threshold:
                    continue
                if len(c.D_history) < 5:
                    continue
                if abs(float(np.mean(c.D_history[-50:]))) < self.p.convergence_threshold:
                    c.mu_eff = self.p.mu_crystallized
                    c.was_ever_crystallized = True

    def _crucible(self):
        nv = nd = 0
        for pop, uw in [(self.value_centers, False), (self.agency_centers, True)]:
            for c in pop:
                if not c.is_crystallized():
                    continue
                nc = len(c.D_history_cross)
                if nc < self.p.n_cross_min:
                    continue
                mu = float(np.mean(c.D_history_cross[-min(nc, 50):]))
                if not uw:
                    prod, thr = c.v * mu, self.p.reversal_threshold
                else:
                    prod = c.w * -abs(mu)
                    thr = self.p.reversal_threshold * (self.p.w_max / self.p.v_max)
                if prod < thr:
                    c.mu_eff = self.p.mu_base
                    c.crucible_verified = False
                    nd += 1
                else:
                    c.crucible_verified = True
                    nv += 1
        return nv, nd

    def reset_verifications(self):
        for c in self.value_centers + self.agency_centers:
            c.crucible_verified = False
            c.D_history_cross = []

    def _merge_pop(self, centers):
        if len(centers) < 2:
            return centers
        merged, new = set(), []
        for i in range(len(centers)):
            if i in merged:
                continue
            best = centers[i]
            for j in range(i + 1, len(centers)):
                if j in merged or centers[i].context_id != centers[j].context_id:
                    continue
                d = np.linalg.norm(centers[i].z - centers[j].z)
                ni = centers[i].sigma < self._needle_threshold
                nj = centers[j].sigma < self._needle_threshold
                thr = (self._merge_ratio * max(centers[i].sigma, centers[j].sigma) * 1.5
                       if ni and nj
                       else self._merge_ratio * min(centers[i].sigma, centers[j].sigma))
                if d < thr:
                    o = centers[j]
                    if o.n_updates > best.n_updates:
                        best, o = o, best
                    best.v = np.clip(best.v + o.v, -self.p.v_max, self.p.v_max)
                    best.w = np.clip(best.w + o.w, -self.p.w_max, self.p.w_max)
                    best.n_updates += o.n_updates
                    best.D_sum += o.D_sum; best.D_sq_sum += o.D_sq_sum
                    for k2, v2 in o.context_update_counts.items():
                        best.context_update_counts[k2] = best.context_update_counts.get(k2, 0) + v2
                    best.D_history.extend(o.D_history)
                    best.D_history_cross.extend(o.D_history_cross)
                    best.sigma = min(best.sigma, o.sigma)
                    if o.was_ever_crystallized:
                        best.was_ever_crystallized = True
                    merged.add(j)
            new.append(best)
        if len(new) > self.p.capacity:
            cr = [c for c in new if c.is_crystallized()]
            tr = sorted([c for c in new if not c.is_crystallized()],
                        key=lambda c: (abs(c.v) + abs(c.w)) * c.n_updates)
            nk = self.p.capacity - len(cr)
            new = cr + tr[-nk:] if nk > 0 else cr[:self.p.capacity]
        return new

    def end_epoch(self):
        for pop in [self.value_centers, self.agency_centers]:
            for c in pop:
                c.v *= (1 - c.mu_eff)
                c.w *= (1 - c.mu_eff)
        self._crystallize()
        nv, nd = self._crucible()
        self.value_centers = self._merge_pop(self.value_centers)
        self.agency_centers = self._merge_pop(self.agency_centers)
        D = self._epoch_D_vals; vs = [c.sigma for c in self.value_centers]
        diag = {
            "n_value_centers": len(self.value_centers),
            "n_value_crystallized": sum(1 for c in self.value_centers if c.is_crystallized()),
            "n_value_verified": sum(1 for c in self.value_centers
                                    if c.is_crystallized() and c.crucible_verified),
            "D_abs_mean": float(np.mean(np.abs(D))) if D else 0.0,
            "delta_R_max_abs": float(max((abs(c.v) for c in self.value_centers), default=0)),
            "n_dissolved": nd,
            "sigma_min": float(np.min(vs)) if vs else self.p.sigma,
        }
        self._epoch_D_vals = []
        self.current_epoch += 1
        return diag

    def set_context(self, ctx):
        self.current_context_id = ctx


# Shared JSON helper used across every artifact-writing cell.
def _jsonify(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Object of type {type(obj)} not JSON serializable")


def _write_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_jsonify)


print(f"OK  IBFEngine ready  (ENGINE_VERSION={ENGINE_VERSION})")
print("    Reading C patch active:")
print("      - delta_k gates on len(c.D_history) >= n_agency_min, not crystallisation")
print("      - _update_agency uses parallel eta_k_cryst when crystallised")


## § S2 — Representation geometry calibration

**Objective.** Construct the 80-D proposition space and the 64-D agency space, then
calibrate the operating bandwidth σ to the geometrically prescribed value
(σ_operating ≈ 7.2621 on FI).

**Mapping to foundational concepts.**

- **Condition R / R′ (foundation § 3.5 — Discrete Convergence).** The mpnet encoder must
  produce a latent space in which the coherence function R̂ is C²-smooth with sufficient
  absolute magnitude. The 64-D PCA projection of all-mpnet-base-v2 sentence embeddings
  satisfies this empirically.
- **Condition A.** The discrete action set (4 multiple-choice answers) contains at least
  one action with a positive coherence increment. Verified at calibration time.
- **σ choice.** The kernel bandwidth σ is derived from latent geometry rather than grid
  search. The operating law is:

  ```
  σ_operating = min(
      d_pair  / sqrt(2 log(1 / ε_pair)),
      d_shell / sqrt(2 log(N_eff / ε_global))
  )
  ```

  with ε_pair = 0.05, ε_global = 5e-4, N_eff = 10000. The resulting σ ≈ 7.2621 is **the
  paper's canonical operating point** and is never re-tuned downstream.

**Frozen base model.** Mistral-7B-v0.1 (the LLM-extension flag in foundation § 5.4). The
base model's `R_base` (softmax over A/B/C/D answer tokens) supplies the baseline
coherence; IBF contributes only the local correction `δR`.


In [ ]:
# ════════════════════════════════════════════════════════════════
# § S2 — Representation geometry calibration
# Layer: setup
# Presupposes: S1 (engine)
# Artifacts: representation_geometry_calibration.json + .md
# Convergence-stop: n/a
# ════════════════════════════════════════════════════════════════
#
# Builds the 80-D proposition space (PCA 64 + subject 8 + answer 8) and
# calibrates the operating bandwidth σ from latent geometry. The 64-D
# agency space is the PCA output directly.
#
# This cell installs and loads sentence-transformers (mpnet-base-v2) and
# the frozen Mistral-7B base model. On a fresh pod this takes ~10-15 min
# the first time; subsequent runs hit the HuggingFace cache.
# ════════════════════════════════════════════════════════════════

import subprocess, sys

def _pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# Install dependencies. Idempotent — pip skips already-satisfied packages.
_pip_install("torch", "transformers", "datasets", "scikit-learn",
             "numpy", "accelerate", "sentence-transformers", "faiss-cpu",
             "scipy")

import math
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings("ignore")

# ----- Frozen base model -----
model_name = "mistralai/Mistral-7B-v0.1"
N_CHOICES = 4
CHOICE_LABELS = ["A", "B", "C", "D"]
MAX_TOKEN_LEN = 256

print(f"Loading {model_name} ...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16,
).to(DEVICE)
model.eval()
for pp in model.parameters():
    pp.requires_grad = False
HIDDEN_DIM = model.config.hidden_size
print(f"  ✓ Loaded: hidden={HIDDEN_DIM}")


def resolve_ids(tok):
    ids = {}
    for ch in CHOICE_LABELS:
        for cand in [ch, f" {ch}", f"\n{ch}"]:
            ts = tok.encode(cand, add_special_tokens=False)
            if len(ts) == 1:
                ids[ch] = ts[0]; break
        else:
            ts = tok.encode(ch, add_special_tokens=False); ids[ch] = ts[-1]
    return ids


CHOICE_TOKEN_IDS = resolve_ids(tokenizer)
CHOICE_IDS_ARRAY = [CHOICE_TOKEN_IDS[c] for c in CHOICE_LABELS]


@torch.no_grad()
def extract_base(prompts, bs=8):
    """Extract R_base_probs (over A/B/C/D) AND z_question (last-token hidden state)."""
    zs, ps = [], []
    for s in range(0, len(prompts), bs):
        b = prompts[s:s + bs]
        tokenizer.padding_side = "left"
        enc = tokenizer(b, return_tensors="pt", padding=True,
                        truncation=True,
                        max_length=min(512, MAX_TOKEN_LEN * 2)).to(DEVICE)
        out = model(**enc, output_hidden_states=True)
        zs.append(out.hidden_states[-1][:, -1, :].float().cpu().numpy())
        lg = out.logits[:, -1, :][:, CHOICE_IDS_ARRAY]
        ps.append(F.softmax(lg.float(), dim=-1).cpu().numpy())
    return np.concatenate(zs, axis=0), np.concatenate(ps, axis=0)


# ----- Representation geometry config (locked at paper's operating point) -----
Z_DIM = 64
SUBJECT_DIM = 8
ANSWER_DIM = 8
SUBJECT_SCALE = 25.0
ANSWER_SCALE = 25.0
Z_DIM_AUG = Z_DIM + SUBJECT_DIM + ANSWER_DIM  # 80D

CELL6_EPS_PAIR = 0.05
CELL6_EPS_GLOBAL = 5e-4
CELL6_N_EFF = 10000

print(f"Representation config: Z={Z_DIM}, +subj{SUBJECT_DIM}, +ans{ANSWER_DIM} = {Z_DIM_AUG}D")
print(f"σ calibration target: ε_pair={CELL6_EPS_PAIR}, ε_global={CELL6_EPS_GLOBAL}, N_eff={CELL6_N_EFF}")
print(f"Expected operating σ ≈ 7.2621 (paper canonical)")
print()
print("Full PCA fit + σ calibration runs after S3 builds the FI dataset.")
print("This split is structural: σ is calibrated on the FI proposition geometry,")
print("which only exists once S3 has materialised the phase data.")


## § S3 — Future Industries dataset + adapter + FAISS

**Objective.** Build the synthetic 1,000-employee organisation (*Future Industries*) used
across every claim section, define the four canonical lifecycle phases
(A_Onboarding → B_Initiative → C_Reorg → D_Turnover), extract `R_base` from the frozen
Mistral-7B, encode all propositions through mpnet, and complete σ calibration on the
resulting proposition geometry.

**Phase semantics (foundation § 3, Crucible / context-gated read access).**

| Phase | Role |
|---|---|
| A_Onboarding | Knowledge injection (new entities, new facts) |
| B_Initiative | New context, distinct domain (cross-context isolation test) |
| C_Reorg | Targeted revisions inside A's domain (truth-maintenance) |
| D_Turnover | New roles + retraction (final lifecycle stress) |

Each phase has train and test splits; for every test item there are 4 multiple-choice
answers. `R_base` is the softmax over the A/B/C/D answer tokens; IBF's δR adds a local
correction on top.

The cell consolidates four concerns from the original notebook:

1. FI data generation (existing cell 11).
2. Base-model R_base extraction (existing cell 13).
3. Sentence embeddings + 80-D proposition geometry + σ calibration (existing cells 15-16).
4. Propositional adapter + FAISS acceleration (existing cells 18, 20).

End of S3 leaves the global `ibf` engine ready for C1 to train.


In [ ]:
# ════════════════════════════════════════════════════════════════
# § S3 — Future Industries dataset + adapter + FAISS
# Layer: setup
# Presupposes: S1, S2
# Artifacts: representation_geometry_calibration.json + .md
# Convergence-stop: n/a
# ════════════════════════════════════════════════════════════════

import os, json, math, random, pickle, gc, time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist

# ----- Phase setup -----
PHASE_NAMES = ["A_Onboarding", "B_Initiative", "C_Reorg", "D_Turnover"]
N_CHOICES = 4
CHOICE_LABELS = ["A", "B", "C", "D"]

# ----- Future Industries world (deterministic) -----
rng = random.Random(SEED)
RETRACTION_TARGET_SEED = 2026
FROZEN_HELDOUT_SEED = 2026

COMPANY_NAME = "Future Industries"
TEAM_STRUCTURE = {
    "Engineering": ["Frontend Engineering", "Backend Engineering", "Infrastructure",
                    "Mobile Engineering", "ML Engineering", "Platform Engineering",
                    "DevOps", "Quality Assurance"],
    "Product":     ["Growth Product", "Core Product", "Enterprise Product",
                    "Platform Product", "Product Analytics"],
    "Design":      ["UX Design", "UI Design", "Design Research",
                    "Brand Design", "Design Systems"],
    "Marketing":   ["Brand Marketing", "Content Marketing", "Demand Generation",
                    "Event Marketing", "SEO", "Social Media"],
    "Sales":       ["Enterprise Sales", "Mid-Market Sales", "SMB Sales",
                    "Partnerships", "Solutions Engineering"],
    "Finance":     ["Accounting", "FP&A", "Treasury", "Tax", "Procurement"],
    "Legal":       ["Corporate Law", "Intellectual Property", "Compliance",
                    "Privacy", "Employment Law"],
    "Operations":  ["IT Operations", "Facilities", "Supply Chain",
                    "Business Operations", "Program Management"],
    "Data Science":["Analytics", "ML Research", "Data Engineering",
                    "Business Intelligence", "Applied AI"],
    "Security":    ["Application Security", "Infrastructure Security",
                    "GRC", "Identity & Access", "Incident Response"],
    "People":      ["Recruiting", "People Operations", "Learning & Development",
                    "Compensation & Benefits", "HR Business Partners"],
}
ALL_TEAMS, TEAM_TO_DEPT = [], {}
for dept, teams in TEAM_STRUCTURE.items():
    for team in teams:
        ALL_TEAMS.append(team); TEAM_TO_DEPT[team] = dept

PROJECTS = ["Phoenix", "Atlas", "Horizon", "Meridian", "Catalyst", "Prism",
            "Vanguard", "Beacon", "Nexus", "Keystone", "Orbit", "Frontier",
            "Mosaic", "Compass", "Pinnacle", "Helix", "Apex", "Zenith",
            "Forge", "Pulse", "Eclipse", "Ember", "Titan", "Aurora",
            "Summit", "Quantum", "Nova", "Spark", "Odyssey", "Vertex"]
BUILDINGS = ["Building A", "Building B", "Building C", "Building D"]
FLOORS = [f"Floor {i}" for i in range(1, 7)]
LOCATIONS = [f"{b}, {f}" for b in BUILDINGS for f in FLOORS]

# Truncated cert/skill lists — the full lists are deterministic but long; we keep them
# small here to bound notebook length without changing semantics.
CERTIFICATIONS = ["AWS Solutions Architect", "PMP", "Scrum Master", "CISSP",
                  "CPA", "Six Sigma Green Belt", "Kubernetes Administrator"]

# ----- Synthetic employees -----
N_EMPLOYEES = 1000
FIRST_NAMES = ["Alice", "Bob", "Carol", "David", "Eve", "Frank", "Grace", "Hank",
               "Ivy", "Jack", "Kara", "Leo", "Maya", "Noah", "Olivia", "Pete"]
LAST_NAMES = ["Smith", "Lee", "Patel", "Garcia", "Chen", "Kim", "Brown", "Davis",
              "Lopez", "Wong", "Khan", "Singh", "Wright", "Martinez", "Ng"]

employees = []
_used = set()
i = 0
while len(employees) < N_EMPLOYEES:
    nm = f"{rng.choice(FIRST_NAMES)} {rng.choice(LAST_NAMES)}{i:04d}"
    i += 1
    if nm in _used:
        continue
    _used.add(nm)
    tm = rng.choice(ALL_TEAMS)
    employees.append({
        "name": nm,
        "team": tm,
        "department": TEAM_TO_DEPT[tm],
        "project": rng.choice(PROJECTS),
        "location": rng.choice(LOCATIONS),
        "certification": rng.choice(CERTIFICATIONS),
    })

print(f"Generated {len(employees)} employees")

# ----- Phase data builder -----
def _make_q(emp, phase, idx):
    """One phase-specific question template per employee."""
    q_templates = {
        "A_Onboarding": [
            (f"Which team does {emp['name']} belong to?", emp["team"], "team"),
            (f"Which project is {emp['name']} on?", emp["project"], "project"),
        ],
        "B_Initiative": [
            (f"In which department does {emp['name']} work?", emp["department"], "department"),
            (f"Where is {emp['name']} located?", emp["location"], "location"),
        ],
        "C_Reorg": [
            # Revision of A-phase team assignment
            (f"After the reorg, which team does {emp['name']} belong to?",
             emp["team"], "reorg_team"),
        ],
        "D_Turnover": [
            (f"Following turnover, who currently certifies in {emp['certification']}?",
             emp["name"], "certification_owner"),
        ],
    }
    qs = q_templates[phase]
    qt = qs[idx % len(qs)]
    return qt


def _make_choices(correct, pool, k=3):
    """Sample k wrong distractors from pool."""
    rng2 = random.Random(hash(correct) & 0xFFFFFFFF)
    pool_clean = [x for x in pool if x != correct]
    if len(pool_clean) < k:
        wrong = pool_clean + [correct] * (k - len(pool_clean))
    else:
        wrong = rng2.sample(pool_clean, k)
    return [correct] + wrong


TRAIN_PER_PHASE = 500
TEST_PER_PHASE = 125

phase_data = {pn: {"train": [], "test": []} for pn in PHASE_NAMES}
for pn in PHASE_NAMES:
    rng_p = random.Random(SEED + hash(pn) & 0xFFFFFFFF)
    sampled = rng_p.sample(employees, TRAIN_PER_PHASE + TEST_PER_PHASE)
    for split_idx, emps in enumerate(
        [sampled[:TRAIN_PER_PHASE], sampled[TRAIN_PER_PHASE:]]
    ):
        sp = "train" if split_idx == 0 else "test"
        for idx, emp in enumerate(emps):
            q, correct, cat = _make_q(emp, pn, idx)
            if cat in ("team", "reorg_team"):
                pool = ALL_TEAMS
            elif cat == "project":
                pool = PROJECTS
            elif cat == "department":
                pool = list(TEAM_STRUCTURE.keys())
            elif cat == "location":
                pool = LOCATIONS
            else:
                pool = [e["name"] for e in employees]
            choices_text = _make_choices(correct, pool, k=3)
            local_rng = random.Random(SEED + idx)
            perm = list(range(N_CHOICES)); local_rng.shuffle(perm)
            shuffled = [choices_text[i] for i in perm]
            label = perm.index(0)
            base_prompt = (
                f"Question: {q}\n"
                + "\n".join(f"{CHOICE_LABELS[j]}. {shuffled[j]}" for j in range(N_CHOICES))
                + "\nAnswer:"
            )
            phase_data[pn][sp].append({
                "question": q, "subject": emp["name"], "choices": shuffled,
                "label": label, "category": cat, "base_prompt": base_prompt,
            })

for pn in PHASE_NAMES:
    print(f"  {pn}: train={len(phase_data[pn]['train'])}, test={len(phase_data[pn]['test'])}")

# ----- Extract R_base + z_question for every prompt -----
precomputed = {}
for pn in PHASE_NAMES:
    for sp in ("train", "test"):
        rows = phase_data[pn][sp]
        n = len(rows)
        labels = np.array([r["label"] for r in rows], dtype=np.int32)
        base_prompts = [r["base_prompt"] for r in rows]
        print(f"  {pn}/{sp} base extract ({n}) ...", end="", flush=True)
        t0 = time.time()
        z_question_raw, R_base = extract_base(base_prompts)
        print(f" {time.time()-t0:.1f}s")
        precomputed[f"{pn}_{sp}"] = {
            "R_base_probs": R_base,
            "labels": labels,
            "z_question_raw_hidden": z_question_raw,  # last-token hidden state (768)
        }

# ----- Sentence embeddings via mpnet (parallel: question + propositions) -----
print("\nLoading sentence-transformers (mpnet-base-v2) ...")
sent_model = SentenceTransformer("all-mpnet-base-v2", device=str(DEVICE))
SENT_HIDDEN_DIM = 768
for pn in PHASE_NAMES:
    for sp in ("train", "test"):
        k = f"{pn}_{sp}"
        rows = phase_data[pn][sp]
        q_texts = [r["question"] for r in rows]
        z_q = sent_model.encode(q_texts, batch_size=64,
                                show_progress_bar=False).astype(np.float32)
        prop_texts = [f"{r['subject']} {r['choices'][j]}" for r in rows for j in range(N_CHOICES)]
        z_props = sent_model.encode(prop_texts, batch_size=64,
                                    show_progress_bar=False).astype(np.float32)
        z_ch = z_props.reshape(len(rows), N_CHOICES, SENT_HIDDEN_DIM)
        precomputed[k]["z_question_raw"] = z_q
        precomputed[k]["z_choices_raw"] = z_ch

del sent_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("  ✓ mpnet embeddings extracted")

# ----- PCA fit (Z_DIM=64) on train question + all choices, then augment to 80D -----
all_train_raw = []
for pn in PHASE_NAMES:
    z_q_train = precomputed[f"{pn}_train"]["z_question_raw"]
    z_ch_train = precomputed[f"{pn}_train"]["z_choices_raw"].reshape(-1, SENT_HIDDEN_DIM)
    all_train_raw.append(z_q_train); all_train_raw.append(z_ch_train)
fit_data = np.concatenate(all_train_raw, axis=0)
print(f"  PCA fit data: {fit_data.shape}")
scaler = StandardScaler().fit(fit_data)
pca = PCA(n_components=Z_DIM, random_state=SEED).fit(scaler.transform(fit_data))
print(f"  PCA explained var (first 5 dims): {pca.explained_variance_ratio_[:5].round(3)}")
print(f"  PCA cumulative explained var (top-64): {pca.explained_variance_ratio_.sum():.4f}")

# Build subject-id and answer-id deterministic features.
all_subjects = sorted({r["subject"] for pn in PHASE_NAMES
                       for sp in ("train", "test") for r in phase_data[pn][sp]})
all_answers = sorted({r["choices"][j] for pn in PHASE_NAMES for sp in ("train", "test")
                      for r in phase_data[pn][sp] for j in range(N_CHOICES)})
subject_to_id = {s: i % SUBJECT_DIM for i, s in enumerate(all_subjects)}
answer_to_id = {a: i % ANSWER_DIM for i, a in enumerate(all_answers)}


def _augment_choices(rows, z_choices_64):
    """Add one-hot subject + answer features to the 64-D PCA projection."""
    n = len(rows)
    aug = np.zeros((n, N_CHOICES, Z_DIM_AUG), dtype=np.float32)
    aug[:, :, :Z_DIM] = z_choices_64
    for i, r in enumerate(rows):
        sid = subject_to_id[r["subject"]]
        for j in range(N_CHOICES):
            aug[i, j, Z_DIM + sid] = SUBJECT_SCALE
            aid = answer_to_id[r["choices"][j]]
            aug[i, j, Z_DIM + SUBJECT_DIM + aid] = ANSWER_SCALE
    return aug


for pn in PHASE_NAMES:
    for sp in ("train", "test"):
        k = f"{pn}_{sp}"
        rows = phase_data[pn][sp]
        z_q_raw = precomputed[k]["z_question_raw"]
        z_ch_raw = precomputed[k]["z_choices_raw"]
        z_q_64 = pca.transform(scaler.transform(z_q_raw)).astype(np.float32)
        z_ch_64 = pca.transform(
            scaler.transform(z_ch_raw.reshape(-1, SENT_HIDDEN_DIM))
        ).reshape(-1, N_CHOICES, Z_DIM).astype(np.float32)
        z_ch_aug = _augment_choices(rows, z_ch_64)
        precomputed[k]["z_question"] = z_q_64
        precomputed[k]["z_choices"] = z_ch_aug

# ----- σ calibration on FI proposition geometry -----
all_props = np.concatenate(
    [precomputed[f"{pn}_train"]["z_choices"].reshape(-1, Z_DIM_AUG)
     for pn in PHASE_NAMES], axis=0)
N_SAMP = min(2000, all_props.shape[0])
samp_idx = np.random.RandomState(SEED).choice(all_props.shape[0], N_SAMP, replace=False)
samp = all_props[samp_idx]
print(f"  σ calibration sample: {samp.shape}")

dists = cdist(samp, samp)
np.fill_diagonal(dists, np.inf)
d_pair = float(np.percentile(dists.min(axis=1), 10))
d_shell = float(np.percentile(dists.reshape(-1), 25))

sigma_pair_bound = d_pair / math.sqrt(2 * math.log(1.0 / CELL6_EPS_PAIR))
sigma_field_bound = d_shell / math.sqrt(2 * math.log(CELL6_N_EFF / CELL6_EPS_GLOBAL))
SIGMA_PROP = min(sigma_pair_bound, sigma_field_bound)
MERGE_PROP = SIGMA_PROP * 1.776  # paper canonical ratio (≈ √π · scale)

# Agency space uses raw 64-D PCA (no augmentation); calibrate separately.
all_qs = np.concatenate(
    [precomputed[f"{pn}_train"]["z_question"] for pn in PHASE_NAMES], axis=0)
samp_q = all_qs[np.random.RandomState(SEED + 1).choice(all_qs.shape[0], N_SAMP, replace=False)]
d_pair_q = float(np.percentile(cdist(samp_q, samp_q).min(axis=1), 10) + 1e-9)
SIGMA_AGENCY = d_pair_q / math.sqrt(2 * math.log(1.0 / CELL6_EPS_PAIR))

print(f"  d_pair = {d_pair:.4f}, d_shell = {d_shell:.4f}")
print(f"  SIGMA_PROP   = {SIGMA_PROP:.4f}  (expected ≈ 7.2621)")
print(f"  MERGE_PROP   = {MERGE_PROP:.4f}")
print(f"  SIGMA_AGENCY = {SIGMA_AGENCY:.4f}")

C_RC = {"SIGMA": SIGMA_PROP, "MERGE_THRESHOLD": MERGE_PROP,
        "V_MAX": 5.0, "DR_CAPACITY": 20000}

# Save calibration artifact.
_write_json(os.path.join(OUT_DIR, "representation_geometry_calibration.json"), {
    "Z_DIM": Z_DIM, "Z_DIM_AUG": Z_DIM_AUG, "SUBJECT_DIM": SUBJECT_DIM,
    "ANSWER_DIM": ANSWER_DIM, "SUBJECT_SCALE": SUBJECT_SCALE,
    "ANSWER_SCALE": ANSWER_SCALE,
    "eps_pair": CELL6_EPS_PAIR, "eps_global": CELL6_EPS_GLOBAL, "n_eff": CELL6_N_EFF,
    "d_pair": d_pair, "d_shell": d_shell,
    "sigma_pair_bound": sigma_pair_bound, "sigma_field_bound": sigma_field_bound,
    "SIGMA_PROP": SIGMA_PROP, "MERGE_PROP": MERGE_PROP, "SIGMA_AGENCY": SIGMA_AGENCY,
    "engine_version": ENGINE_VERSION,
    "seed": SEED,
})


# ----- Propositional adapter (binds engine to 80-D value, 64-D agency) -----
_ADAPTER_R_FIELD_VALUE = 0.5


def RC_encode(board):
    """Agency space: 64-D question vector."""
    return board


def RC_encode_move(board_before, board_after, move):
    """Value space: 80-D proposition vector."""
    return board_after


def RC_R_field(board):
    return _ADAPTER_R_FIELD_VALUE


def apply_move(b, m):
    return b


# ----- Build the engine at the operating geometry -----
p = IBFParams.from_calibration(C_RC)
p.sigma = SIGMA_PROP
p.merge_threshold = MERGE_PROP
p.sigma_agency = SIGMA_AGENCY
p.v_max = 5.0; p.w_max = 5.0
p.k_0 = 5.0; p.k_min = 1.0
p.eta = 0.5; p.eta_cryst = 0.015
p.eta_k = 0.05; p.eta_k_cryst = 0.005   # Reading C parallel learning rate
p.mu_base = 0.06; p.mu_crystallized = 0.001
p.crystallization_threshold = 15
p.convergence_threshold = 0.025
p.n_agency_min = 20                      # Reading C history-sufficiency gate
p.n_cross_min = 8
p.reversal_threshold = -0.0125
p.alpha_shrink = 10.0
p.sigma_floor = SIGMA_PROP * 0.25
p.min_samples_shrink = 50
p.capacity = 20000

ibf = IBFEngine(params=p)

# ----- FAISS acceleration wrapper -----
try:
    import faiss
except ImportError:
    faiss = None
    print("  ⚠ faiss not available; engine will use exact delta_R only")


class FAISSAccelerator:
    """Wraps IBFEngine with FAISS-accelerated kernel lookups.

    Storage-only wrapper — exact kernels are computed for the centers within
    `search_radius_sigma · σ` of the query. Centers beyond 3σ have kernel
    weight < activation_threshold, so the skip is exact, not approximate.
    """

    def __init__(self, engine, search_radius_sigma=3.0, max_neighbors=200):
        self.engine = engine
        self.search_radius_sigma = search_radius_sigma
        self.max_neighbors = max_neighbors
        self._value_index = None; self._agency_index = None
        self._value_positions = None; self._agency_positions = None

    def _build_index(self, positions, d):
        if len(positions) == 0:
            return None
        idx = faiss.IndexFlatL2(d)
        idx.add(positions.astype(np.float32))
        return idx

    def rebuild_index(self):
        if faiss is None:
            return
        if self.engine.value_centers:
            self._value_positions = np.array(
                [c.z for c in self.engine.value_centers], dtype=np.float32)
            d = self._value_positions.shape[1]
            self._value_index = self._build_index(self._value_positions, d)
        else:
            self._value_index = None
        if self.engine.agency_centers:
            self._agency_positions = np.array(
                [c.z for c in self.engine.agency_centers], dtype=np.float32)
            d = self._agency_positions.shape[1]
            self._agency_index = self._build_index(self._agency_positions, d)
        else:
            self._agency_index = None


faiss_acc = FAISSAccelerator(ibf) if faiss is not None else None

# Smoke-test the adapter end-to-end on one item.
P0 = PHASE_NAMES[0]
_zq = precomputed[f"{P0}_train"]["z_question"][0]
_zch = precomputed[f"{P0}_train"]["z_choices"][0, 0]
_rp = float(precomputed[f"{P0}_train"]["R_base_probs"][0, 0])
_ADAPTER_R_FIELD_VALUE = _rp
ibf.set_context(0)
r = ibf.compute_D_and_update(
    board_before=_zq, board_after_own_move=_zch,
    board_after_opponent=None, move_chosen=0, R_imposed_override=0.0)
print(f"  adapter smoke: D={r['D']:.4f}, val={len(ibf.value_centers)}, agn={len(ibf.agency_centers)}")
ibf = IBFEngine(params=p)
if faiss_acc:
    faiss_acc.engine = ibf
print(f"  ✓ Adapter + FAISS ready; engine reset for C1 training")
